# تحليل بيانات Fridge-tag 2E
استخراج البيانات من ملفات PDF الخاصة بأجهزة Fridge-tag 2E وتحويلها إلى جداول قابلة للتحليل.

**ملاحظة:** قبل رفع هذا الملف إلى المستودع (commit)، يجب مسح جميع المخرجات عبر `Kernel → Restart & Clear Output`.

In [ ]:
# 1. الاستيرادات الأساسية وتعريف المسارات
import sys
import re
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pdfplumber

# تعريف المسارات كمتغيرات (لتسهيل التعديل لاحقاً)
DATA_DIR = Path("data")
INPUT_FT2_DIR = DATA_DIR / "input_ft2"
INPUT_RAW_DIR = DATA_DIR / "input_raw"
EXTRACTED_TEXT_DIR = DATA_DIR / "extracted_texts"
OUTPUT_TABLES_DIR = DATA_DIR / "output_tables"

# إنشاء المجلدات إذا لم تكن موجودة
EXTRACTED_TEXT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("✅ البيئة:", sys.executable)
print("✅ pandas:", pd.__version__)
print("✅ numpy:", np.__version__)

In [ ]:
# 2. دالة تحويل المدة إلى دقائق (معالجة جميع الصيغ: أيام، ساعات، دقائق)
def duration_to_minutes(dur_str):
    """
    تحويل صيغ المدة مثل:
    '0min', '11h 44min', '1d 08:59h', '1d 08h 59min', '704', '1440', '19min' ... إلى دقائق (int)
    """
    if pd.isna(dur_str) or dur_str == '':
        return 0
    dur_str = str(dur_str).strip()
    
    # إذا كان الرقم فقط (مثل '704' أو '1440')
    if dur_str.isdigit():
        return int(dur_str)
    
    total = 0
    
    # استخراج الأيام (d)
    if 'd' in dur_str:
        days = re.search(r'(\d+)d', dur_str)
        if days:
            total += int(days.group(1)) * 1440
    
    # استخراج الساعات (h)
    if 'h' in dur_str:
        # حالة مثل '08:59h'
        if ':' in dur_str:
            parts = dur_str.split('h')[0]
            h, m = map(int, parts.split(':'))
            total += h * 60 + m
        else:
            # حالة مثل '11h' أو '1h'
            hours = re.search(r'(\d+)h', dur_str)
            if hours:
                total += int(hours.group(1)) * 60
    
    # استخراج الدقائق (min)
    if 'min' in dur_str:
        mins = re.search(r'(\d+)\s*min', dur_str)
        if mins:
            total += int(mins.group(1))
    
    return total

In [ ]:
# 3. مثال: معالجة ملف PDF واحد (للتوضيح)
sample_pdf = INPUT_FT2_DIR / "130600112663_202603251042.pdf"

if sample_pdf.exists():
    with pdfplumber.open(sample_pdf) as pdf:
        print(f"عدد الصفحات: {len(pdf.pages)}")
        first_page_text = pdf.pages[0].extract_text()
        print("أول 500 حرف من النص:")
        print(first_page_text[:500])
else:
    print(f"الملف {sample_pdf} غير موجود. تأكد من المسار.")

In [ ]:
# 4. استخراج الجداول من ملف PDF واحد (تجريبي)
if sample_pdf.exists():
    all_tables = []
    with pdfplumber.open(sample_pdf) as pdf:
        for i, page in enumerate(pdf.pages):
            tables = page.extract_tables()
            for table in tables:
                if table:
                    df = pd.DataFrame(table)
                    df['source_page'] = i + 1
                    all_tables.append(df)
    
    if all_tables:
        print(f"عدد الجداول المستخرجة: {len(all_tables)}")
        display(all_tables[0].head())
    else:
        print("لم يتم العثور على جداول.")
else:
    print(f"الملف {sample_pdf} غير موجود.")

In [ ]:
# 5. معالجة جميع ملفات PDF في مجلد input_ft2 (إكمال الحلقة غير المكتملة سابقاً)
print("بدء معالجة جميع ملفات PDF...")

for pdf_file in INPUT_FT2_DIR.glob("*.pdf"):
    print(f"\n--- معالجة: {pdf_file.name} ---")
    
    # استخراج النص الكامل
    full_text = ""
    with pdfplumber.open(pdf_file) as pdf:
        for page in pdf.pages:
            txt = page.extract_text()
            if txt:
                full_text += txt + "\n"
    
    # حفظ النص في ملف txt
    txt_output = EXTRACTED_TEXT_DIR / f"{pdf_file.stem}.txt"
    with open(txt_output, "w", encoding="utf-8") as f:
        f.write(full_text)
    print(f"  ✓ تم حفظ النص في: {txt_output}")
    
    # استخراج الجداول
    all_tables = []
    with pdfplumber.open(pdf_file) as pdf:
        for i, page in enumerate(pdf.pages):
            tables = page.extract_tables()
            for table in tables:
                if table:
                    df = pd.DataFrame(table)
                    df['source_page'] = i + 1
                    all_tables.append(df)
    
    # حفظ الجداول في ملف Excel
    if all_tables:
        excel_output = OUTPUT_TABLES_DIR / f"{pdf_file.stem}_tables.xlsx"
        with pd.ExcelWriter(excel_output) as writer:
            for idx, df in enumerate(all_tables):
                sheet_name = f"Table_{idx+1}"
                # تقصير اسم الورقة إذا كان طويلاً (Excel يسمح بـ 31 حرفًا كحد أقصى)
                if len(sheet_name) > 31:
                    sheet_name = sheet_name[:31]
                df.to_excel(writer, sheet_name=sheet_name, index=False)
        print(f"  ✓ تم حفظ {len(all_tables)} جدول في: {excel_output}")
    else:
        print(f"  ⚠ لم يتم العثور على جداول في {pdf_file.name}")

print("\n✅ انتهت معالجة جميع الملفات.")

In [ ]:
# 6. تحويل النص المستخرج إلى DataFrame منظم (لملف نموذجي)
if sample_pdf.exists():
    txt_path = EXTRACTED_TEXT_DIR / f"{sample_pdf.stem}.txt"
    if txt_path.exists():
        with open(txt_path, "r", encoding="utf-8") as f:
            lines = f.readlines()
        
        data_rows = []
        for line in lines:
            # تم تعديل (\d+min) إلى (\S+) ليدعم صيغ الوقت المختلفة أسوة بالعمود الآخر
            match = re.match(r'(\d+)\s+(\d{2}\.\d{2}\.\d{4})\s+(\S*)\s+([+-]?\d+\.\d+°C)\s+(\S+)\s+([+-]?\d+\.\d+°C)\s+(\S+)\s+(\S*)\s+([+-]?\d+\.\d+°C)\s+(\S+)\s+(\S*)', line)
            if match:
                data_rows.append(match.groups())
        
        if data_rows:
            columns = ["No.", "Date", "Events", "Avg Temp", "Status_Low", "Min Temp", "Time_Below", "Alarm_Trigger_Low",
                       "Max Temp", "Time_Above", "Alarm_Trigger_Upper"]
            df_parsed = pd.DataFrame(data_rows, columns=columns)
            
            # تحويل درجات الحرارة إلى أرقام
            for col in ["Avg Temp", "Min Temp", "Max Temp"]:
                df_parsed[col] = df_parsed[col].str.replace("°C", "").astype(float)
            
            # تحويل المدد إلى دقائق باستخدام الدالة الصحيحة (تم توحيد الطريقة للعمودين)
            df_parsed["Time_Above_min"] = df_parsed["Time_Above"].apply(duration_to_minutes)
            
            # Time_Below_min: محفوظ للاستخدام المستقبلي
            # السبب:
            # - دالة duration_to_minutes مصممة أصلاً لمعالجة كلا الاتجاهين (أعلى وأدنى)
            # - في أنظمة مراقبة سلسلة التبريد الدوائية (Fridge-tag 2E) يُطلب عادة تحليل الانحرافات المنخفضة (< 2°C أو الحد الأدنى المسموح)
            # - يُمكن استخدامه لاحقاً في فلترة الإنذارات المنخفضة، تقارير الامتثال الثنائي، أو رسوم بيانية مقارنة
            # - الاحتفاظ به الآن يتجنب إعادة الحساب مستقبلاً ويحافظ على اكتمال معالجة البيانات الخام
            df_parsed["Time_Below_min"] = df_parsed["Time_Below"].apply(duration_to_minutes)
            
            display(df_parsed.head())
        else:
            print("لم يتم العثور على صفوف بيانات مطابقة للنمط.")
    else:
        print(f"ملف النص {txt_path} غير موجود. قم بتشغيل الخلية السابقة أولاً.")
else:
    print(f"الملف {sample_pdf} غير موجود.")

In [ ]:
# 7. رسم بياني: تطور أقصى درجة حرارة يومية (إذا توفرت البيانات)
if 'df_parsed' in locals() and not df_parsed.empty:
    df_parsed["Date_dt"] = pd.to_datetime(df_parsed["Date"], format="%d.%m.%Y")
    
    plt.figure(figsize=(12, 5))
    plt.plot(df_parsed["Date_dt"], df_parsed["Max Temp"], marker='o', linestyle='-', linewidth=1)
    plt.axhline(y=8.0, color='r', linestyle='--', label='الحد الأعلى (+8°C)')
    plt.xlabel("التاريخ")
    plt.ylabel("أقصى درجة حرارة (°C)")
    plt.title("تطور أقصى درجة حرارة يومية - جهاز Fridge-tag")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("لا توجد بيانات كافية للرسم. تأكد من معالجة الملف أولاً.")

In [ ]:
# 8. تصفية الأيام التي استوفت شرط الإنذار (>= 600 دقيقة فوق 8°C)
if 'df_parsed' in locals() and not df_parsed.empty:
    alarm_days = df_parsed[df_parsed["Time_Above_min"] >= 600]
    if not alarm_days.empty:
        print("أيام الإنذار (انحراف حراري طويل):")
        display(alarm_days[["Date", "Avg Temp", "Max Temp", "Time_Above_min"]])
    else:
        print("لا توجد أيام بها انحراف حراري طويل (>= 600 دقيقة).")
    
    short_excursions = df_parsed[(df_parsed["Max Temp"] > 8.0) & (df_parsed["Time_Above_min"] < 600)]
    if not short_excursions.empty:
        print("\nانحرافات قصيرة (< 600 دقيقة):")
        display(short_excursions[["Date", "Max Temp", "Time_Above_min"]])
else:
    print("لا توجد بيانات. قم بتشغيل الخلايا السابقة أولاً.")